# CLV Thesis — Tuned Performance Workflow (Dunnhumby)

Supplementary analysis to the fixed-parameter thesis evidence set: **how good can the joint
models become in practice** on the dense Dunnhumby panel when hyperparameters are tuned?
The frozen thesis results (`experiments/final_manifest.yaml`) are untouched; everything here
is a separate, clearly-labeled evidence set.

Protocol:

1. **Grid HPO with an honest validation window** (`tune.py --hpo-val-pct 0.2`): trials train on
   the first 64 calibration weeks and are scored on the last 16 calibration weeks; the true
   holdout is never seen during hyperparameter selection. Four studies: Joint LSTM and Joint
   Transformer on the main 80/22 protocol, plus the two full-covariate Extension 3 models on the
   80/4 protocol. Grids: LSTM lr x hidden_size x dropout (18 trials); Transformer
   lr x (d_model, n_heads) (15 trials). Single seed 42 per trial, reduced fidelity
   (max_epochs 80, 20 scenarios).
2. **Winner transfer** (`scripts/make_tuned_configs.py`): each winner's hyperparameters are
   stamped onto the frozen `configs_final` counterpart — only HP keys and the epoch cap may
   change (asserted). The Extension 3 full-covariate winner is transferred to ALL FOUR covariate
   variants, so the tuned ablation compares covariate sets under identical hyperparameters.
3. **Tuned finals**: 10 configs x 3 seeds (42/7/2024), sample-mode inference with 30 scenarios —
   identical evaluation to the fixed protocol. The epoch cap is lifted to 250 because every fixed
   Dunnhumby run terminated at the 150-epoch cap with early stopping never firing; patience-20
   early stopping on the calibration-internal validation split decides the actual length.
4. **Full-cohort SHAP** on the tuned full-covariate checkpoints, explaining the *identical*
   100 background / 701 explained households as the thesis run (committed sample manifests in
   `experiments/shap_samples/`), at the same fixed 128-integration-sample budget.

Stage toggles `RUN_HPO / RUN_FINALS / RUN_SHAP` (Cell 3) allow splitting across sessions if the
~12h GPU cap binds. Requires **GPU T4 x2** (T4 x1 works but roughly doubles HPO/finals time).
Download `results_archive_tuned.zip`, unzip locally to `results/final_kaggle_tuned/`, then run
`python build_tuned_results.py`.


In [ ]:
import subprocess, sys, os, shutil
from pathlib import Path

# ── 1. Install missing packages (skipped if already present) ─────────────────
# Do NOT touch numpy (Kaggle base image ships NumPy 2.x; codebase is compatible).
def _need_install(pkg_name):
    try:
        __import__(pkg_name)
        return False
    except ImportError:
        return True

to_install = []
if _need_install("lifetimes"):  to_install.append("lifetimes>=0.11.3")
if _need_install("openpyxl"):   to_install.append("openpyxl>=3.1.0")
if _need_install("optuna"):     to_install.append("optuna>=3.0.0")

if to_install:
    print(f"Installing: {to_install}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + to_install, check=True)
    print("Done.\n")
else:
    print("All packages already installed.\n")

# ── 2. Kaggle environment flag (train.py / run_seeds.py path remapping) ──────
os.environ["KAGGLE_ENV"] = "1"
print("KAGGLE_ENV=1 set.\n")

# ── 3. Clone (or refresh) the repo from GitHub ────────────────────────────────
# Internet must be ON. Pin a commit via os.environ["THESIS_REF"] = "<sha>".
REPO_URL  = "https://github.com/OttoPrins/thesis-code-final.git"
REPO_PATH = Path("/kaggle/working/thesis-code")
REPO_REF  = os.environ.get("THESIS_REF", "main")

if REPO_PATH.exists():
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "--all", "--tags", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "checkout", REPO_REF, "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", f"origin/{REPO_REF}", "--quiet"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_PATH)], check=True)

os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))
sha = subprocess.check_output(
    ["git", "-C", str(REPO_PATH), "rev-parse", "--short", "HEAD"]
).decode().strip()
print(f"Repo : {REPO_PATH}  @ {sha}  (ref={REPO_REF})")

# ── 4. Stage the Dunnhumby dataset (only dataset this workflow needs) ─────────
DATA_ROOT = Path("/kaggle/working/input")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

dh_target  = DATA_ROOT / "dunnhumby"
dh_mounted = Path("/kaggle/input/dunnhumby")
if dh_target.exists():
    print("  dunnhumby: already present.")
elif dh_mounted.exists():
    dh_target.symlink_to(dh_mounted)
    print("  dunnhumby: symlinked from /kaggle/input/.")
else:
    print("  dunnhumby: not mounted — downloading ...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "ottoprins/dunnhumby",
         "-p", str(dh_target), "--unzip"],
        check=True,
    )
    print("  dunnhumby: download complete.")

os.environ["KAGGLE_DATA_ROOT"] = str(DATA_ROOT)
print(f"KAGGLE_DATA_ROOT={DATA_ROOT}\n")

# The SHAP pipeline (scripts/run_extension3_shap.py) resolves the config's
# repo-relative raw_dir directly (no Kaggle override), so symlink it to the mount.
raw_dir = REPO_PATH / "data/raw/Dunnhumby datasets"
raw_dir.parent.mkdir(parents=True, exist_ok=True)
if raw_dir.is_symlink():
    raw_dir.unlink()
elif raw_dir.exists():
    shutil.rmtree(raw_dir)
raw_dir.symlink_to(dh_target.resolve(), target_is_directory=True)
print(f"SHAP raw_dir symlink: {raw_dir} -> {dh_target.resolve()}\n")

# ── 5. Verify GPU (fail fast if incompatible) ─────────────────────────────────
import torch
print(f"torch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Enable one: Settings -> Accelerator -> GPU T4 x2.")
n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    name = torch.cuda.get_device_name(i)
    cap  = torch.cuda.get_device_capability(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i}: {name}  (sm_{cap[0]}{cap[1]}, {vram:.1f} GB)")
    if cap < (7, 0):
        raise RuntimeError(
            f"{name} has CUDA capability sm_{cap[0]}{cap[1]} (need sm_70+). "
            "Fix: Settings -> Accelerator -> GPU T4 x2, then restart the kernel."
        )
if n_gpus < 2:
    print("WARNING: single GPU session — HPO and finals run serially (~2x wall time).")

# ── 6. Output directory ───────────────────────────────────────────────────────
Path("/kaggle/working/results").mkdir(parents=True, exist_ok=True)
print("\n/kaggle/working/results/ — ready.")


In [ ]:
import os, subprocess, sys
from pathlib import Path

data_root = Path(os.environ.get("KAGGLE_DATA_ROOT", "/kaggle/input"))
required = ["transaction_data.csv", "hh_demographic.csv", "campaign_table.csv",
            "campaign_desc.csv", "coupon_redempt.csv"]
dh = data_root / "dunnhumby"
missing = [f for f in required if not (dh / f).exists()]
if missing:
    raise SystemExit(f"Missing Dunnhumby files in {dh}: {missing}")
print(f"OK  {dh}: {required}")

# Committed SHAP sample manifests (identical households as the thesis run).
for name in ["observed_demographics_seed42_100x701.json",
             "all_households_seed42_100x100.json"]:
    p = Path("experiments/shap_samples") / name
    if not p.exists():
        raise SystemExit(f"Missing committed SHAP sample manifest: {p}")
print("OK  experiments/shap_samples manifests present.")

print("\nRunning Dunnhumby pipeline validation ...")
subprocess.run([sys.executable, "validate_pipelines.py", "--dataset", "dunnhumby"], check=True)


In [ ]:
from pathlib import Path

# Stage toggles — flip to split a timed-out session. Later stages need the
# earlier stages' outputs in /kaggle/working/results of the SAME session
# (or a re-uploaded copy).
RUN_HPO    = True
RUN_FINALS = True
RUN_SHAP   = True

SEEDS = ["42", "7", "2024"]
HEARTBEAT_INTERVAL = "60"

# HPO settings: honest validation window + reduced trial fidelity (grid mode).
HPO_VAL_PCT     = 0.2     # last 20% of calibration weeks = HPO validation window
HPO_MAX_EPOCHS  = "80"
HPO_N_SCENARIOS = "20"

RESULTS_DIR = Path("/kaggle/working/results")
WINNER_DIR  = RESULTS_DIR / "hpo_winners"
WINNER_DIR.mkdir(parents=True, exist_ok=True)

HPO_STUDIES = [
    {"name": "lstm_joint_dunnhumby",
     "config": "experiments/configs_final/lstm_joint_dunnhumby_final.yaml"},
    {"name": "transformer_joint_dunnhumby",
     "config": "experiments/configs_final/transformer_joint_dunnhumby_final.yaml"},
    {"name": "extension3_lstm_full_dunnhumby",
     "config": "experiments/configs_final/extension3_lstm_full_dunnhumby_final.yaml"},
    {"name": "extension3_transformer_full_dunnhumby",
     "config": "experiments/configs_final/extension3_transformer_full_dunnhumby_final.yaml"},
]
for s in HPO_STUDIES:
    s["out"] = str(WINNER_DIR / f"{s['name']}_winner.yaml")

# Tuned finals: epoch cap lifted (fixed runs all hit the 150 cap; ES decides).
TUNED_MAX_EPOCHS = "250"
TUNED_CONFIG_DIR = "experiments/configs_tuned"
tuned_configs = [
    "lstm_joint_dunnhumby_tuned",
    "transformer_joint_dunnhumby_tuned",
    "extension3_lstm_none_dunnhumby_tuned",
    "extension3_lstm_static_dunnhumby_tuned",
    "extension3_lstm_dynamic_dunnhumby_tuned",
    "extension3_lstm_full_dunnhumby_tuned",
    "extension3_transformer_none_dunnhumby_tuned",
    "extension3_transformer_static_dunnhumby_tuned",
    "extension3_transformer_dynamic_dunnhumby_tuned",
    "extension3_transformer_full_dunnhumby_tuned",
]

print(f"HPO studies : {len(HPO_STUDIES)} (grid, hpo-val-pct={HPO_VAL_PCT})")
print(f"Tuned finals: {len(tuned_configs)} configs x {len(SEEDS)} seeds")
print(f"Stages      : HPO={RUN_HPO}  FINALS={RUN_FINALS}  SHAP={RUN_SHAP}")


In [ ]:
# Pareto/NBD on Dunnhumby (fast, deterministic). Re-run here so the tuned
# archive is self-contained: its arrays.npz doubles as the raw-holdout ground
# truth used by build_tuned_results.py for monetary rescoring.
import subprocess, sys

subprocess.run(
    [sys.executable, "run_benchmarks.py",
     "--config", "experiments/configs_final/lstm_base_dunnhumby_final.yaml",
     "--models", "pareto_nbd"],
    check=True,
)
print("\nPareto/NBD benchmark complete.")


In [ ]:
# Grid HPO: two tune.py workers per study share one Optuna SQLite study
# (GridSampler distributes trials atomically across workers). After both
# workers exit, a finalize pass (--n-trials 0) rewrites the winner from the
# COMPLETE study — this closes the race where the first-finishing worker
# writes a winner while the other worker's last trial is still running.
import os, subprocess, sys, time
from pathlib import Path
import torch

def _worker_cmd(study, db, extra):
    return [
        sys.executable, "-u", "tune.py",
        "--config", study["config"],
        "--sampler", "grid",
        "--storage", f"sqlite:///{db}",
        "--study-name", f"{study['name']}_val20",
        "--hpo-val-pct", str(HPO_VAL_PCT),
        "--max-epochs", HPO_MAX_EPOCHS,
        "--n-scenarios", HPO_N_SCENARIOS,
        "--out-config", study["out"],
        "--kaggle",
    ] + extra

if RUN_HPO:
    n_workers = min(2, torch.cuda.device_count())
    for study in HPO_STUDIES:
        db = RESULTS_DIR / f"hpo_{study['name']}_val20.db"
        t0 = time.time()
        print(f"\n{'='*70}\nHPO study: {study['name']}  ({n_workers} worker(s))\n{'='*70}", flush=True)
        procs = []
        for w in range(n_workers):
            env = os.environ.copy()
            env["CUDA_VISIBLE_DEVICES"] = str(w)
            env["PYTHONUNBUFFERED"] = "1"
            procs.append(subprocess.Popen(_worker_cmd(study, db, []), env=env))
        codes = [p.wait() for p in procs]
        if any(c != 0 for c in codes):
            raise RuntimeError(f"HPO worker failed for {study['name']}: exit codes {codes}")
        # Finalize: recompute the winner from the complete shared study.
        Path(study["out"]).unlink(missing_ok=True)
        subprocess.run(_worker_cmd(study, db, ["--n-trials", "0"]), check=True)
        assert Path(study["out"]).exists(), f"Winner config missing: {study['out']}"
        print(f"[{study['name']}] done in {(time.time()-t0)/60:.1f} min -> {study['out']}", flush=True)
    print("\nAll HPO studies complete.")
else:
    for study in HPO_STUDIES:
        assert Path(study["out"]).exists(), (
            f"RUN_HPO=False but winner config missing: {study['out']} — "
            "restore it from a previous session's archive first."
        )
    print("HPO stage skipped (winner configs already present).")


In [ ]:
# Stamp each study winner's hyperparameters onto the frozen configs_final
# counterparts. Only HP keys + epoch cap + run_name may differ (asserted).
import shutil, subprocess, sys
from pathlib import Path

subprocess.run(
    [sys.executable, "scripts/make_tuned_configs.py",
     "--winner_main_lstm",         HPO_STUDIES[0]["out"],
     "--winner_main_transformer",  HPO_STUDIES[1]["out"],
     "--winner_ext3_lstm",         HPO_STUDIES[2]["out"],
     "--winner_ext3_transformer",  HPO_STUDIES[3]["out"],
     "--out_dir", TUNED_CONFIG_DIR,
     "--max_epochs", TUNED_MAX_EPOCHS],
    check=True,
)

# Archive the winners + tuned configs with the results for full provenance.
shutil.copytree(TUNED_CONFIG_DIR, RESULTS_DIR / "configs_tuned", dirs_exist_ok=True)
n_cfg = len(list(Path(TUNED_CONFIG_DIR).glob("*.yaml")))
assert n_cfg == len(tuned_configs), f"Expected {len(tuned_configs)} tuned configs, found {n_cfg}"
print(f"\n{n_cfg} tuned configs written and archived to {RESULTS_DIR / 'configs_tuned'}.")


In [ ]:
# Tuned finals: 10 configs x 3 seeds, sample mode (30 scenarios, from config).
# No --skip_existing: that gate matches the FROZEN fixed-parameter manifest and
# would never recognise tuned runs.
import subprocess, sys

if RUN_FINALS:
    cmd = [
        sys.executable, "run_seeds.py",
        "--config_dir", TUNED_CONFIG_DIR,
        "--configs", *tuned_configs,
        "--seeds", *SEEDS,
        "--modes", "sample",
        "--heartbeat_interval", HEARTBEAT_INTERVAL,
    ]
    subprocess.run(cmd, check=True)
    print("\nTuned multi-seed sweep complete.")
else:
    print("Finals stage skipped.")


In [ ]:
# Full-cohort SHAP on the tuned full-covariate checkpoints. The committed
# sample manifests pin the identical 100 background / 701 explained households
# used by the thesis run, so tuned vs fixed attribution is directly comparable.
# 128 integration samples = the budget the thesis' convergence audit escalated
# to; supplying it directly is the conservative choice for the tuned models.
import json, shutil, subprocess, sys
from pathlib import Path

if RUN_SHAP:
    samples_dir = RESULTS_DIR / "shap" / "samples"
    samples_dir.mkdir(parents=True, exist_ok=True)
    for name in ["observed_demographics_seed42_100x701.json",
                 "all_households_seed42_100x100.json"]:
        shutil.copy2(Path("experiments/shap_samples") / name, samples_dir / name)
    print(f"Sample manifests staged in {samples_dir}")

    cmd = [
        sys.executable, "scripts/run_extension3_shap.py",
        "--results_root", str(RESULTS_DIR),
        "--checkpoint_root", str(RESULTS_DIR / "checkpoints"),
        "--output_root", str(RESULTS_DIR / "shap"),
        "--device", "cuda",
        "--analysis_seed", "42",
        "--n_background", "100",
        "--n_explain", "701",
        "--sensitivity_n_explain", "100",
        "--fixed_integration_samples", "128",
        "--n_bootstrap", "2000",
        "--lstm_config", f"{TUNED_CONFIG_DIR}/extension3_lstm_full_dunnhumby_tuned.yaml",
        "--transformer_config", f"{TUNED_CONFIG_DIR}/extension3_transformer_full_dunnhumby_tuned.yaml",
        "--lstm_run_prefix", "extension3_lstm_full_dunnhumby_tuned",
        "--transformer_run_prefix", "extension3_transformer_full_dunnhumby_tuned",
    ]
    print("Starting tuned full-cohort SHAP (expected ~2-4 h).")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

    # Verify: 16 rows (2 arch x 2 heads x 4 features), identical households.
    import pandas as pd
    summary = pd.read_csv(RESULTS_DIR / "tables" / "shap_extension3_summary.csv")
    assert len(summary) == 16, f"Expected 16 SHAP summary rows, found {len(summary)}"
    assert set(summary["n_households"]) == {701}
    assert set(summary["n_integration_samples"]) == {128}
    staged = json.loads((samples_dir / "observed_demographics_seed42_100x701.json").read_text())
    committed = json.loads(Path("experiments/shap_samples/observed_demographics_seed42_100x701.json").read_text())
    assert staged == committed, "SHAP sample manifest drifted from the committed thesis manifest."
    print("\nTuned SHAP complete — 16 summary rows, households identical to the thesis run.")
else:
    print("SHAP stage skipped.")


In [ ]:
# In-session sanity aggregation. --include_exploratory bypasses the frozen
# fixed-parameter manifest gate (tuned config hashes are not in it by design).
# The authoritative tables are built LOCALLY afterwards by build_tuned_results.py.
import json, subprocess, sys
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "src.evaluation.compare",
     "--results_dir", str(RESULTS_DIR),
     "--latex", "--seeds",
     "--include_exploratory",
     "--protocol_variant", "all"],
    check=False,
)

# Quick tuned-vs-fixed headline check. Fixed references = frozen thesis values
# (results/thesis_final_v2, Dunnhumby 80/22, mean over seeds 42/7/2024).
FIXED_REFERENCE = {
    "lstm_joint_dunnhumby":        {"freq_mape": 5.07, "bias_pct": 0.87,  "spend_r2_log": 0.44, "clv_spearman": 0.81},
    "transformer_joint_dunnhumby": {"freq_mape": 5.89, "bias_pct": 2.37,  "spend_r2_log": 0.50, "clv_spearman": 0.82},
}
PARETO_REFERENCE = {"freq_mape": 74.21, "bias_pct": -74.21, "spend_r2_log": -2.23, "clv_spearman": 0.28}

print(f"\n{'='*78}")
print("Tuned vs fixed (Dunnhumby 80/22, tuned = mean over seeds; fixed = thesis values)")
print(f"{'='*78}")
print("Pareto/NBD reference:", PARETO_REFERENCE)
for stem, fixed in FIXED_REFERENCE.items():
    vals = {k: [] for k in fixed}
    for seed in SEEDS:
        p = RESULTS_DIR / "tables" / f"{stem}_tuned_seed{seed}_sample_metrics.json"
        if not p.exists():
            print(f"  {stem}_tuned: missing {p.name} — finals incomplete?")
            break
        m = json.loads(p.read_text())
        for k in vals:
            vals[k].append(float(m.get(k, float("nan"))))
    else:
        tuned = {k: sum(v) / len(v) for k, v in vals.items()}
        print(f"\n  {stem}:")
        for k in fixed:
            print(f"    {k:<14} fixed={fixed[k]:>8.3f}   tuned={tuned[k]:>8.3f}   delta={tuned[k]-fixed[k]:>+8.3f}")
print("\nNote: deltas are informative only — the authoritative comparison (rescored")
print("monetary metrics + paired bootstrap) is produced locally by build_tuned_results.py.")


In [ ]:
# Verify completeness, then archive everything for download.
import json, shutil
from pathlib import Path

tables = RESULTS_DIR / "tables"
expected_runs = [f"{cfg}_seed{seed}_sample" for cfg in tuned_configs for seed in SEEDS]

problems = []
for run in expected_runs:
    mp = tables / f"{run}_metrics.json"
    if not mp.exists():
        problems.append(f"missing metrics: {mp.name}")
        continue
    m = json.loads(mp.read_text())
    if not m.get("run_valid", False):
        problems.append(f"run_valid=False: {mp.name} ({m.get('run_invalid_reason', '?')})")
    if not (tables / f"{run}_arrays.npz").exists():
        problems.append(f"missing arrays: {run}_arrays.npz")
    if not (RESULTS_DIR / "checkpoints" / f"{run}.pt").exists():
        problems.append(f"missing checkpoint: {run}.pt")

if not (tables / "pareto_nbd_dunnhumby_arrays.npz").exists():
    problems.append("missing pareto_nbd_dunnhumby_arrays.npz (ground truth for rescoring)")
if RUN_SHAP and not (tables / "shap_extension3_summary.csv").exists():
    problems.append("missing shap_extension3_summary.csv")

if problems:
    print("INCOMPLETE RUN — archive anyway, but note:")
    for p in problems:
        print(f"  - {p}")
else:
    print(f"All {len(expected_runs)} tuned runs valid; benchmark + SHAP outputs present.")

n_metrics = len(list(tables.glob("*_metrics.json")))
n_ckpt = len(list((RESULTS_DIR / "checkpoints").glob("*.pt")))
print(f"\nMetrics files: {n_metrics} | Checkpoints: {n_ckpt}")

archive = shutil.make_archive("/kaggle/working/results_archive_tuned", "zip", RESULTS_DIR)
size_mb = Path(archive).stat().st_size / 1024**2
print(f"\nArchive: {archive} ({size_mb:.1f} MB)")
print("Download via Kaggle Output tab, then locally:")
print("  unzip results_archive_tuned.zip -d results/final_kaggle_tuned")
print("  python build_tuned_results.py")
